# 02 ML baseline

A conventional global LightGBM model built with Nixtla MLForecast: sales lags, rolling means, Fourier terms and calendar features. We start on the raw target, see where it breaks, and settle on the reference model for the rest of Day 1. Later notebooks change one decision at a time.

Runs in about a minute: two models refit at ten origins.


In [1]:
import os
import sys
import warnings

if "google.colab" in sys.modules:
    !git clone -q https://github.com/tackes/time-series-forecasting-bootcamp
    %cd time-series-forecasting-bootcamp
    %pip install -q statsforecast==2.1.1 mlforecast==1.1.0 utilsforecast==0.2.17 lightgbm==4.7.0 catboost==1.2.10
elif not os.path.exists("shared"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from mlforecast.target_transforms import LocalStandardScaler

from shared.prepare_data import load_panel, series_segments, H, FREQ, ALL_CUTOFFS, DEV_CUTOFFS, TEST_CUTOFF
from shared.features import LGB_PARAMS, sin1, cos1, sin2, cos2
from shared.leaderboard import to_long, save_forecasts, load_forecasts, leaderboard, breakdown

panel = load_panel()
actuals = panel[["unique_id", "ds", "y"]]

## Features

One model is trained on all 2,000 series. Each training row is a series-week with features built only from earlier weeks.

- **Lags** of sales: 1, 2, 3, 4, 8 and 13 weeks back.
- **Rolling means** of the last 4, 13 and 26 weeks, computed from lag 1.
- **Calendar**: month, and two annual Fourier pairs from the day of year.

There is no lag 52. A year-ago lag would need 53+ weeks per series, which removes most of the training rows at the early origins. Annual pattern comes from the Fourier terms.

Fourier terms here are functions of the date, so a given week has the same value for every series and every cutoff. They are known for any future week, so there is nothing to leak. Each is a one-line function in `shared/features.py`, for example `sin1(dates) = np.sin(2 * np.pi * dates.dayofyear / 365.25)`; `sin2` and `cos2` use `4 * np.pi`.

Later notebooks import these same settings from `shared/features.py` as `LAGS`, `LAG_TRANSFORMS`, `DATE_FEATURES` and `LGB_PARAMS`.

In [3]:
lags = [1, 2, 3, 4, 8, 13]
lag_transforms = {1: [RollingMean(window_size=4), RollingMean(window_size=13), RollingMean(window_size=26)]}
date_features = ["month", sin1, cos1, sin2, cos2]
LGB_PARAMS

{'n_estimators': 300,
 'learning_rate': 0.05,
 'num_leaves': 31,
 'min_child_samples': 50,
 'subsample': 0.8,
 'subsample_freq': 1,
 'colsample_bytree': 0.8,
 'random_state': 0,
 'verbosity': -1}

In [4]:
mlf_raw = MLForecast(
    models={"lgbm_raw": lgb.LGBMRegressor(**LGB_PARAMS)},
    freq=FREQ,
    lags=lags,
    lag_transforms=lag_transforms,
    date_features=date_features,
    num_threads=1,
)

## Fit and forecast one origin

We fit on data up to the cutoff only. `predict(h=14)` is recursive: it forecasts week 1, appends that forecast to the history, recomputes the lags and rolling means, and forecasts week 2. By week 14 the lag-1 feature is the model's own forecast, never an observed value from the target period.

In [5]:
train = actuals[actuals.ds <= TEST_CUTOFF]
mlf_raw.fit(train)
preds = mlf_raw.predict(H)
preds.head()

,unique_id,ds,lgbm_raw
0,10006_S1,2025-04-16,1500.560565
1,10006_S1,2025-04-23,1548.124116
2,10006_S1,2025-04-30,1597.147512
3,10006_S1,2025-05-07,1651.954490
4,10006_S1,2025-05-14,1693.877024


In [6]:
mlf_raw.preprocess(train).dropna().head(3).T

,26,27,28
unique_id,10006_S1,10006_S1,10006_S1
ds,2024-01-03 00:00:00,2024-01-10 00:00:00,2024-01-17 00:00:00
y,2050.0,1720.0,1600.0
lag1,1560.0,2050.0,1720.0
lag2,900.0,1560.0,2050.0
lag3,1400.0,900.0,1560.0
lag4,3000.0,1400.0,900.0
lag8,1440.0,1540.0,1810.0
lag13,1630.0,1280.0,1870.0
rolling_mean_lag1_window_size4,1715.0,1477.5,1557.5


## A scaled version

Series differ in volume by orders of magnitude, and a tree model on raw units spends its splits on the big series. Small and intermittent series fall into leaves shared with them.
The fix here is a per-series standardization of the target: `LocalStandardScaler` subtracts each series' mean and divides by its standard deviation before training, and reverses that on the forecasts. The mean and standard deviation come from the training data at each cutoff.
Notebook 04 compares scaling choices properly. For now this is the model every later notebook starts from, called `lgbm_baseline`.

In [7]:
mlf = MLForecast(
    models={"lgbm_baseline": lgb.LGBMRegressor(**LGB_PARAMS)},
    freq=FREQ,
    lags=lags,
    lag_transforms=lag_transforms,
    date_features=date_features,
    target_transforms=[LocalStandardScaler()],
    num_threads=1,
)

## Every origin

Both models are refit at each cutoff, using only data up to that cutoff.

In [8]:
forecasts = []
for cutoff in ALL_CUTOFFS:
    train = actuals[actuals.ds <= cutoff]
    for model in [mlf_raw, mlf]:
        model.fit(train)
        forecasts.append(to_long(model.predict(H), cutoff, "02_ml_baseline"))

forecasts = pd.concat(forecasts)

## Compared with the statistical baselines

Same series, same weeks, same metrics.

In [9]:
columns = ["model", "coverage", "wape", "accuracy", "bias", "mean_rmsse", "median_rmsse", "apc", "apc_median"]
baselines = load_forecasts(["01_baselines"])
candidates = pd.concat([baselines, forecasts])

leaderboard(candidates, actuals, scope="dev")[columns].round(3)

,model,coverage,wape,accuracy,bias,mean_rmsse,median_rmsse,apc,apc_median
0,Average,1.000,0.377,0.623,-0.063,1.662,0.850,0.119,0.123
1,SeasonalNaive,0.988,0.408,0.592,-0.042,1.850,1.055,0.000,0.000
2,HistoricAverage,1.000,0.426,0.574,-0.060,1.791,0.887,0.015,0.016
3,lgbm_baseline,1.000,0.449,0.551,0.139,1.865,0.921,0.120,0.127
4,Naive,1.000,0.454,0.546,-0.088,1.860,0.969,0.352,0.370
5,lgbm_raw,1.000,0.455,0.545,0.044,5.501,1.224,0.203,0.223


In [10]:
leaderboard(candidates, actuals, scope="test")[columns].round(3)

,model,coverage,wape,accuracy,bias,mean_rmsse,median_rmsse,apc,apc_median
0,Average,1.0,0.384,0.616,0.006,1.275,0.726,0.109,0.101
1,lgbm_baseline,1.0,0.419,0.581,0.109,1.302,0.733,0.092,0.076
2,Selected_per_SKU,1.0,0.425,0.575,0.030,1.366,0.737,NaN,NaN
3,HistoricAverage,1.0,0.426,0.574,-0.011,1.367,0.739,0.010,0.010
4,SeasonalNaive,1.0,0.438,0.562,-0.037,1.525,0.929,0.000,0.000
5,Naive,1.0,0.470,0.530,0.066,1.332,0.771,0.327,0.308
6,lgbm_raw,1.0,0.505,0.495,0.215,12.000,1.065,0.164,0.169


Read the development table first. The final-window table is one 14-week window that we inspected while choosing this recipe (the per-series scaler and the 26-week rolling mean), so treat it as an exploratory benchmark.
Where does the raw model go wrong? Split the development results by sales volume.

In [11]:
segments = series_segments(panel)
volume = breakdown(candidates, actuals, "volume", scope="dev", segments=segments)
volume[volume.model.str.startswith("lgbm")][["model", "volume", "wape", "bias", "mean_rmsse"]].round(3)

,model,volume,wape,bias,mean_rmsse
4,lgbm_baseline,high,0.439,0.145,1.714
5,lgbm_raw,high,0.434,0.021,1.739
10,lgbm_baseline,low,0.737,-0.018,2.065
11,lgbm_raw,low,2.678,2.407,14.861
16,lgbm_baseline,mid,0.543,0.089,1.857
17,lgbm_raw,mid,0.507,0.088,1.718


### Reading the result

- Scaling fixes the low-volume series: development WAPE on them falls from 2.68 to 0.74 and bias from +2.41 to -0.02. Mean RMSSE, which weights every series equally, falls from 5.50 to 1.87.
- Overall WAPE barely moves (0.455 to 0.449). WAPE is dominated by the high-volume series, where the raw model was already as good.
- Neither model beats the simple baselines on the development origins: seasonal naive scores 0.408 and the average of the three baselines 0.377. The scaled model also overforecasts (bias +0.139). Notebooks 03 to 06 work on that.

Try it later: add 52 to `lags`, or remove the Fourier terms from `date_features`, and compare the development table.

In [12]:
save_forecasts(forecasts, "02_ml_baseline")